In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col, to_timestamp, from_utc_timestamp, date_format, round
import pandas as pd
import numpy as np

In [0]:
jdbc_url = dbutils.secrets.get(scope="Capstone", key="DatabasejdbcUrl")#DatabasejdbcUrl
connection_properties = {
    "user": dbutils.secrets.get(scope="Capstone", key="DatabaseUsername"),
    "password": dbutils.secrets.get(scope="Capstone", key="DatabasePassword"),
    "driver": dbutils.secrets.get(scope="Capstone", key="DatabaseDriver")
}

old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Silver.Historical_Prices",
    properties=connection_properties
)

In [0]:
def calculate_rsi_numpy(close_prices, period=14):
    delta = np.diff(close_prices)
    delta = np.concatenate(([np.nan], delta))

    gains = np.where(delta > 0, delta, 0)
    losses = np.where(delta < 0, -delta, 0)

    avg_gains = np.zeros_like(close_prices)
    avg_losses = np.zeros_like(close_prices)
    rsi = np.zeros_like(close_prices, dtype=float)

    if len(close_prices) >= period:
        avg_gains[period-1] = np.mean(gains[1:period+1])
        avg_losses[period-1] = np.mean(losses[1:period+1])
    
    for i in range(period, len(close_prices)):
        avg_gains[i] = (avg_gains[i-1] * (period - 1) + gains[i]) / period
        avg_losses[i] = (avg_losses[i-1] * (period - 1) + losses[i]) / period
    
    with np.errstate(divide='ignore', invalid='ignore'):
        rs = avg_gains / avg_losses
        rsi = np.where(avg_losses != 0, 100 - (100 / (1 + rs)), 100)
    
    rsi[:period-1] = np.nan
    return np.round(rsi, 2).tolist(), np.round(gains, 2).tolist(), np.round(losses, 2).tolist()

period=14
pandas_df = old_data.orderBy("Stock_Symbol", "Date").select("Stock_Symbol", "Date", "Close").toPandas()

pandas_df[f"RSI_{period}"] = np.nan
pandas_df[f"Gains_{period}"] = np.nan
pandas_df[f"Losses_{period}"] = np.nan

for symbol, group in pandas_df.groupby("Stock_Symbol"):
    rsi, gains, losses = calculate_rsi_numpy(group["Close"].to_numpy(), period=period)
    pandas_df.loc[group.index, f"RSI_{period}"] = rsi
    pandas_df.loc[group.index, f"Gains_{period}"] = gains
    pandas_df.loc[group.index, f"Losses_{period}"] = losses

rsi_df = spark.createDataFrame(pandas_df[["Stock_Symbol", "Date", f"RSI_{period}", f"Gains_{period}", f"Losses_{period}"]])

new_data = old_data.join(
    rsi_df,
    on=["Stock_Symbol", "Date"],
    how="left"
)

new_data.show(5)

In [0]:
def calculate_mard_numpy(close_prices, period=14, reference_prices=None):
    if reference_prices is None:
        reference_prices = np.zeros_like(close_prices, dtype=float)
        for i in range(period - 1, len(close_prices)):
            reference_prices[i] = np.mean(close_prices[i - period + 1:i + 1])
        reference_prices[:period - 1] = np.nan

    abs_diff = np.abs(close_prices - reference_prices)
    relative_diff = np.where(
        close_prices != 0,
        abs_diff / close_prices,
        np.nan  # Handle division by zero
    )

    mard = np.zeros_like(close_prices, dtype=float)
    for i in range(period - 1, len(close_prices)):
        mard[i] = np.mean(relative_diff[i - period + 1:i + 1])
    mard[:period - 1] = np.nan
    
    return mard

pandas_df = old_data.orderBy("Stock_Symbol", "Date").select("Stock_Symbol", "Date", "Close").toPandas()

period = 14
pandas_df[f"MARD_{period}"] = pandas_df.groupby("Stock_Symbol")["Close"].transform(
    lambda x: np.round(x.rolling(window=period, min_periods=period).mean(),2)
)

mard_df = spark.createDataFrame(pandas_df[["Stock_Symbol", "Date", f"MARD_{period}"]])

new_data = new_data.join(
    mard_df,
    on=["Stock_Symbol", "Date"],
    how="left"
)

new_data.show(5)

In [0]:
new_data = new_data.orderBy("Stock_Symbol","Date")
display(new_data)